# nano Beatrix bed — the deciding tests again, with AdamW in place of pure Adam

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_bed_adamw.ipynb)

**Why this runs.** The v1 screen (`nano_v1_head_decay.ipynb`, `nano_v1_head_decay_lead_s4.ipynb`) found what makes the last block's big direction: the model's confidence dial, climbing through the final norm against an output layer that keeps growing when nothing pulls it back. With the output layer's readout trained by Adam or AdamW the direction held at 0.23 - 0.32 of the last block's output against 0.92 - 0.93, at no cost in bits, on both of Castor's seeds and on the lead body. Every test that chose the twins' bodies and their frames ran on the bed's pure Adam (learning rate 0.001, betas 0.9 / 0.95, no weight decay). This notebook runs those tests again with AdamW, so the choices are made under the optimizer the crafts will train with.

**The one change.** For a weight p with Adam's moments m and v (bias-corrected), learning rate lr on the run's schedule:

| | the update |
|---|---|
| pure Adam (every earlier run) | p <- p - lr * m / (sqrt(v) + 1e-8) |
| AdamW, the decay set | p <- p * (1 - lr * 0.1) - lr * m / (sqrt(v) + 1e-8) |
| AdamW, every other parameter | p <- p - lr * m / (sqrt(v) + 1e-8) (Adam's update exactly) |

The decay set (`bed_optim`, set `mat`): every weight that maps the stream, the attention projections, the bank's transforms and experts, the Canon kernels, the output layer's readout, its projection and its aleph read (7.95M of Castor's 8.43M parameters). Never decayed: the byte and position tables, the address books, the byte-class amplitude tables, the norms, biases and gates. Everything else is each test's own recipe, verbatim (the same rows, the same steps and windows, the learning rate's warmup and cosine, clip 1.0). Each run lands beside the pure-Adam run of the same settings, under the same name plus `_wd0.1`.

**The parts.** No two parts share a run, so each card can take its own part (open this notebook once per card and set `PART` in C1). Hours at the speeds the earlier runs reached on an RTX PRO 6000 (the splat bodies only ever ran on the 4090, so theirs are upper bounds; C2 measures the card in hand):

| part | what it decides again | what the pure Adam chose | runs | training hours |
|---|---|---|---|---|
| `frames-form` | the frame form: plain, sized or sliced (the fourth pass) | sliced: every close free; Pollux's copy and long window best sliced on both seeds | 24 | 1.9 + 0.4 |
| `bodies-twins` | the curriculum train of Castor, Pollux, Pollux with Canon layers, Pollux's layout with dual hubs and Canon layers (seeds 1, 2) | the strongest seeds of these bodies became the v1 crafts | 8 chains | 2.8 - 6.4 + 0.8 |
| `bodies-splat` | the splat bodies' train (dual hubs in Pollux's layout; four or five dual hubs with softmax blocks; six dual hubs with and without Canon layers) | Pollux's layout with dual hubs became a v1 craft | 10 chains | up to 5.8 - 13.8 + 1.0 |
| `frames-capacity` | records a lookup frame (4, 8, 16) and the scrutiny set (the second pass) | eight records a frame (it forms at 1,024 and carries to 4,096); nesting 0.6, a key defined twice in 0.3 of the rows | 20 | 2.4 + 0.3 |
| `frames-registers` | letters, digits and hex values under their own register tags (the first pass) | the caption register for every pass after it | 12 | 1.5 + 0.2 |
| `frames-pass3` | the scrutiny reads by continuation, sixteen records a frame, Pollux's hub address (the third pass, with the stage-A crafts it continued) | the pair's address (32 x 16) | 37 | 2.8 + 0.6 |
| `pair` | the twins trained together, three forms of the cross term (its solo controls are `bodies-twins`' first blocks) | Castor's guidance at every byte lifted Pollux seed 2's far finding 0.31 -> 0.67 | 12 | 1.9 + 0.2 |

About 23 - 34 hours of card time in all. For three cards, by what each part gates: card 1 `frames-form`, then `frames-capacity`, `frames-registers`, `frames-pass3`; card 2 `bodies-twins`, then `pair`; card 3 `bodies-splat`. The bodies and the pairs train on the sliced frames (the pure Adam's pick); if `frames-form` picks another form under AdamW, the bodies' chains run again in that form (C1: `R.FORM = '<form>'`, a new set of names).

**Token:** `HF_TOKEN` with write access to AbstractPhil/alephllm-nano-beatrix-training (every finished run ships its record and weights under `bed/`, beside the pure-Adam runs).

Run C1 to C3 in order; C4 reads every run beside its pure-Adam twin, any time.

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.31"
import torch
import geolip.nano_beatrix as N
from geolip.nano_beatrix import bed_rerun as R
print('geolip.nano_beatrix', N.__version__, '| torch', torch.__version__, '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 31), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner ships through the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting from the record, its source beside it ============
PART = 'frames-form'          # this card's part, one of: frames-form, bodies-twins, bodies-splat, frames-capacity, frames-registers,
                              # frames-pass3, pair (the head's table; no two parts share a run, so each card takes its own)
WD = 0.1                      # AdamW's decoupled decay: the setting the v1 screen read on Castor's two seeds and the lead body (2026-10-14),
                              # the common setting; 0 gives the pure-Adam runs' own names (their records are read, nothing new trains)
WD_SET = 'mat'                # the decay set: every weight that maps the stream (bed_optim); 'head' = the output layer's two readouts only
OUT = f'/content/bed_adamw_{PART}'                     # the session folder: the corpus, the records, the weights, runner.log
REPO = R.REPO                                          # AbstractPhil/alephllm-nano-beatrix-training: every finished run ships to bed/
PARALLEL = 1                  # runs at once on the card: ONE (a 9M craft at 8 x 4,096 saturates an RTX PRO 6000 alone; four at once ran at
                              # a quarter of the rate each, 2026-10-13)
PRICE_STEPS = 200             # C2's short run of each shape on this card for the price line; never shipped
assert PART in R.ORDER, f'PART is one of {R.ORDER}'
print('decision block: part', PART, '| AdamW decay', WD, 'on the', WD_SET, 'set | out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')
for st in R.parts(WD, WD_SET)[PART]:
    print('  -', st['name'], '(a curriculum train)' if st['kind'] == 'curriculum' else '')

In [ ]:
# C2 (P) — PREFLIGHT: the data, every run the part may train (those already on the Hub marked), the decay set on each body, a short run of
# each shape on this card (the price line), the push probe. Trains nothing of the part.
import json, os, time
from geolip.alephllm.model.alephlm import AlephLM
from geolip.nano_beatrix import bed_corpus, bed_optim, bed_runner
from geolip.nano_beatrix.bodies import craft_cfg

if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', R.CORPUS_BYTES)
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'))

plan = R.plan(PART, WD, WD_SET, repo=REPO)
print(f"the part: {plan['runs']} runs, {plan['chains']} curriculum chains")

SHAPES = R.shapes(PART, WD, WD_SET)
for arm in dict.fromkeys(a for a, _, _ in SHAPES):
    c = bed_optim.census(AlephLM(craft_cfg(arm, 1024)), WD_SET)
    print(f"{arm}: AdamW decay {WD} on {c['decay_tensors']} tensors ({c['decay_params']:,} parameters); {c['other_params']:,} at decay 0")

rate = {}
for arm, ctx, mb in SHAPES:
    spec = dict(arms=arm, seeds='1', ctx=ctx, mb=mb, steps=PRICE_STEPS, smoke=1, log_every=20, register='caption', frames='dyn', plant=0.75,
                frame_records=8, sizes='32,128,256' if ctx <= 1024 else '32,128,256,512,1024', **R.opt_keys(WD, WD_SET))
    res = bed_runner.run(spec, out=OUT)
    assert res['failed'] is None, f'the short run of {arm} at {ctx} x {mb} failed: read the lines above'
    rate[(arm, ctx, mb)] = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]['tok_s'] * 1e3
lo, hi = R.price(PART, lambda a, c, m: rate[(a, c, m)], WD, WD_SET)
units = plan['runs'] + 6 * plan['chains']
print('this card: ' + ', '.join(f'{a} {v / 1e3:.0f}k tok/s at {c:,} x {m}' for (a, c, m), v in rate.items()))
print(f'-> the part trains {lo:.1f} to {hi:.1f} h one run at a time (the curricula as their reads decide), plus about {units / 60:.1f} h for '
      f'the starts, the gauges and the ships; runs already on the Hub are read, not rerun')

PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = f'bed/preflight/push_probe_bed_adamw_{PART}.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0), 'part': PART,
                       'wd': WD, 'wd_set': WD_SET, 'rate': {f'{a} {c}x{m}': v for (a, c, m), v in rate.items()}}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message=f'bed AdamW re-run {PART}: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: C3 runs locally and ships nothing')

In [ ]:
# C3 (T) — THE PART: step after step (a grid of runs, or a curriculum train whose blocks follow its stages' rules), one run at a time on
# the card; every finished run ships in the background as it lands (a Hub fault retries and never stops the part); a STOP file in OUT starts
# no new run. Rerun the cell to continue, in this session or a new one: the runs on disk or on the Hub are read, not rerun. An interrupt
# loses only the running run (the bed has no mid-run checkpoint).
res = R.run(PART, OUT, push=REPO if (HF_TOKEN and PUSH_OK) else '', wd=WD, wd_set=WD_SET, parallel=PARALLEL)
print({k: v for k, v in res.items() if k != 'steps'})
for name, r in res['steps'].items():
    print(' ', name, ':', r)

In [ ]:
# C4 — THE READ: every run beside the pure-Adam run of the same settings (records from this session or the Hub); a curriculum train
# block by block, and its boundary table. Run any time; it trains nothing.
from IPython.display import Markdown, display
from geolip.nano_beatrix import bed_curriculum as bc
display(Markdown(R.read(PART, OUT, REPO, WD, WD_SET)))
for st in R.parts(WD, WD_SET)[PART]:
    if st['kind'] == 'curriculum':
        display(Markdown(f"#### {st['name']}: the boundary table (AdamW)\n\n" + bc.report(st['stages'], st['base'], st['arms'], st['seeds'], OUT, REPO)))

### Reading the result

Each row reads a run's pure-Adam record, then its AdamW record (`a -> b`). The choice a part made under the pure Adam holds when its
rankings hold under AdamW:

| part | the read that decided it |
|---|---|
| `frames-form` | the copy's whole value at 1,024 and at 4,096, and the text bits, plain against sized against sliced, on both twins and both seeds |
| `bodies-twins`, `bodies-splat` | the boundary tables: where the copy formed, how far the long window carried the far records, the text bits after the anneal |
| `frames-capacity` | the whole value by frame size at four, eight and sixteen records; the nested and redefined-key questions |
| `frames-registers` | the copy and the text under each register |
| `frames-pass3` | the scrutiny reads continued at 4,096; sixteen records; Pollux's address sizes |
| `pair` | each paired twin against the same twin trained alone (both under AdamW) |

### Changing the run
| to | the edit |
|---|---|
| another part on this card | C1: `PART = '<part>'`, then C1 to C3 |
| stop after the running run | `open(f'{OUT}/STOP', 'w').close()` (remove the file and rerun C3 to continue) |
| the bodies in another frame form | C1, after the import: `R.FORM = 'plain'` (or `'sized'`) |
| the decay on the readouts only | C1: `WD_SET = 'head'` (its own names) |